## 96. Global Coverage

### 1. Packages

In [1]:
# Packages
import geopandas as gpd
import glob
import os
import rioxarray as rxr
from tqdm import tqdm
from pyproj import Geod
import pandas as pd
from google.cloud import storage

# TODO: align with the computations with the tables in the SQO 
# TODO: Horizontal and vertical coverage; area stats to be normalized w.r.t. IPCC ref region area to compare? 
# TODO: Percentage data output w.r.t. intertidal mask iso tile area? Initially we estimated 9% of the mask so we should be close to this?

C:\Users\kras\AppData\Local\Temp\ipykernel_4624\171936701.py:2: DeprecationWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas still uses PyGEOS by default. However, starting with version 0.14, the default will switch to Shapely. To force to use Shapely 2.0 now, you can either uninstall PyGEOS or set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In the next release, GeoPandas will switch to using Shapely by default, even if PyGEOS is installed. If you only have PyGEOS installed to get speed-ups, this switch should be smooth. However, if you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


### 2. Settings

In [2]:
# Settings
mode = 'intertidal_improved_100m_global'   # Specify mode, either 'intertidal' or 'subtidal'
start_date = '2021-01-01'                  # Start date of the composites
stop_date = '2022-01-01'                   # End date of the composites
compo_int = 12                             # Composite interval [months]
compo_len = 12                             # Composite length [months]
scale = 100                                # Output resolution of the image [m]
crs = 'EPSG:4326'                          # Output projection of the image

upscale = 100                              # Upscaling factor for the image
res_arc_min = 1/16                         # resolution of the image in arc minutes
res_deg = res_arc_min / 60                 # resolution of the image in degrees


# Tiling (see https://www.openearth.nl/rws-bathymetry/2019.html)
zoom_levels = [9, 10, 11] # list with zoom levels
zoom_level = zoom_levels[1] # zoom level to be used

# Directories and files
dir_path_base = r'p:\11209821-cmems-global-sdb'
file_path_tiles = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_polygons_world', f'df_boxes_world_Z{zoom_level}_filtered_v2.parquet') # Tiles file
file_path_sdbs = glob.glob(os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', mode, '05_reprojected', '*.tif'))        # SDB files
file_path_sdbs = [file_path for file_path in file_path_sdbs if not file_path.endswith('.tif.aux.xml')]
file_path_progress = os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', 'progress_global')
print('Number of SDB files:', len(file_path_sdbs))

Number of SDB files: 2235


### 3. Read tiles

In [3]:
# Read tiles
gdf_tiles_org = gpd.read_parquet(file_path_tiles)
gdf_tiles_org['nearest_station_distance'] = gdf_tiles_org['nearest_station_distance']/1000  # Convert to km
gdf_tiles_org['intertidal_coverage'] = gdf_tiles_org['intertidal_coverage']*100  # Convert to percentage
gdf_tiles_org['intertidal_coverage_ed'] = gdf_tiles_org['intertidal_coverage_ed']*100  # Convert to percentage
gdf_tiles = gdf_tiles_org.copy()
gdf_tiles['processed'] = (gdf_tiles['nearest_station_distance'] <= 37) & (gdf_tiles['intertidal_coverage'] >= 1) & (gdf_tiles['intertidal_coverage_ed'] > 0)
gdf_tiles = gdf_tiles[gdf_tiles['processed']]

# Add sdb file_path to tiles
# for i, row in tqdm(gdf_tiles.iterrows(), total=gdf_tiles.shape[0]):
#     file_path = [file_path for file_path in file_path_sdbs if os.path.basename(file_path).startswith(row['name'])]
#     gdf_tiles.at[i, 'file_path'] = file_path[0] if file_path else None

# # Remove tiles without SDB file_path
# gdf_tiles = gdf_tiles[gdf_tiles['file_path'].notna()]

gdf_tiles.head()

,name,id,tx,ty,zoom,geometry,path,ref_region,area_perc_org,area_perc_red,nearest_station_id,nearest_station_longitude,nearest_station_latitude,nearest_station_distance,intertidal_coverage,intertidal_coverage_ed,processed
0,z10_x561_y116,2195,561.0,116.0,10,"POLYGON ((17.22656 79.87430, 17.57813 79.87430...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,2.32,1.18,station 07499,17.33421,79.92993,3.062257,8.286716,1.182026,True
1,z10_x667_y108,13635,667.0,108.0,10,"POLYGON ((54.49219 80.35700, 54.84375 80.35700...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,1.29,0.14,station 07502,54.76023,80.27526,12.472398,16.578475,0.135397,True
2,z10_x566_y141,2760,566.0,141.0,10,"POLYGON ((18.98438 78.20656, 19.33594 78.20656...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,0.71,0.07,station 07487,18.46672,77.95945,35.249050,2.918707,0.157026,True
3,z10_x545_y140,491,545.0,140.0,10,"POLYGON ((11.60156 78.27820, 11.95313 78.27820...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,22.14,21.76,station 07438,11.91355,78.40147,10.215705,22.184213,21.787025,True
7,z10_x550_y146,1037,550.0,146.0,10,"POLYGON ((13.35938 77.84185, 13.71094 77.84185...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,17.38,16.74,station 07491,13.86779,78.09662,25.418648,17.374234,16.777855,True


### 4. Check cloud progress

In [77]:
# open excel file
prog_excel = pd.read_excel(r"p:\11209821-cmems-global-sdb\01_intertidal\02_data\05_calibrated\progress_global\global_progress.xlsx", header=1)

# find index where column complete has a number
last_complete_idx = min(prog_excel["complete"][~prog_excel["complete"].notna()].index)
ref_regions_comp = list(prog_excel["ref_region"].iloc[0:last_complete_idx])

# tiles combined submitted for processing
tiles_submitted = gdf_tiles[gdf_tiles["ref_region"].isin(ref_regions_comp)]
print("Submitted:",len(tiles_submitted)*2)

Submitted: 18830


In [78]:
# stored in the cloud

# cloud variables
source_project = "cmems-sdb-11209821-002"
source_bucket_name = "cmems-isdb"
source_bucket_proj_tiff = "intertidal_improved_100m_global/z10"
source_bucket_proj_meta = "intertidal_improved_100m_global_meta/z10"

# create python storage clients for project
source_client = storage.Client(project=source_project)
source_bucket = source_client.bucket(source_bucket_name)

# List all files under the source prefix
blobs_data = list(source_client.list_blobs(source_bucket, prefix=source_bucket_proj_tiff))
blobs_meta = list(source_client.list_blobs(source_bucket, prefix=source_bucket_proj_meta))
blobs_data_name = [blobs.name[len(source_bucket_proj_tiff)-3:].strip(".tif") for blobs in blobs_data]
blobs_meta_name = [blobs.name[len(source_bucket_proj_meta)-3:].strip(".csv") for blobs in blobs_meta]
print("tiffs:", len(blobs_data_name))
print("meta:", len(blobs_meta_name))

# combined processed (tiffs + csvs)
blobs_proc = blobs_data_name + blobs_meta_name
print("total:", len(blobs_proc))

tiffs: 8768
meta: 9191
total: 17959


In [92]:
# combined successes (tiffs and csvs)
comb_succ = [blobs for blobs in blobs_data_name if blobs in blobs_meta_name]
print("combined succes:", len(comb_succ))

# succes for tiffs and failures for csvs
succ_tiff_fail_csv = [blobs for blobs in blobs_data_name if blobs not in blobs_meta_name]
print("failed csv:",len(succ_tiff_fail_csv))

# succes for csvs and failures for tiffs
succ_csv_fail_tiff = [blobs for blobs in blobs_meta_name if blobs not in blobs_data_name]
print("failed tiff:", len(succ_csv_fail_tiff))

# failures for both tiffs and csvs
blobs_proc_name = [blob.split("/t")[0].replace("/", "_") for blob in blobs_proc]
comb_fail = tiles_submitted["name"][~tiles_submitted["name"].isin(blobs_proc_name)]
print("combined failed:", len(comb_fail), "*2=", len(comb_fail)*2)

# failed in total
print("failed total:", len(tiles_submitted)*2-len(blobs_proc))

combined succes: 8767
failed csv: 1
failed tiff: 424
combined failed: 223 *2= 446
failed total: 871


In [97]:
# build a dataframe of failed tasks (using three options above)
succ_tiff_fail_csv_name = [blob.split("/t")[0].replace("/", "_") for blob in succ_tiff_fail_csv]
succ_csv_fail_tiff_name = [blob.split("/t")[0].replace("/", "_") for blob in succ_csv_fail_tiff]
comb_fail_name = [blob.split("/t")[0].replace("/", "_") for blob in comb_fail]
failed_list = succ_tiff_fail_csv_name + succ_csv_fail_tiff_name + comb_fail_name
print("failed total unique:", len(failed_list))

# make df
failed_df = tiles_submitted[tiles_submitted["name"].isin(failed_list)]
#print(failed_df.shape)
#print(failed_df["ref_region"].value_counts())

failed total unique: 648


### 5. Calculate area based on feasability map

In [32]:
# Reproject tiles to metres
gdf_tiles = gdf_tiles.to_crs('EPSG:6933') # Project to equal-area for grid-based metrics, EPSG 3857 is not to be used for area or dist (only mapping)

# Calculate area of the tiles
gdf_tiles['tile_area'] = gdf_tiles['geometry'].area / 1e6  # Convert to km2

# Calculate coverage
gdf_tiles['intertidal_area_ed'] = gdf_tiles['tile_area'] * gdf_tiles['intertidal_coverage_ed']/100

# # most accurate but need loop
# gdf_tiles = gdf_tiles.to_crs('EPSG:4326')
# # WGS84 ellipsoid
# geod = Geod(ellps="WGS84")
# # Compute area (in square meters), always positive
# area, _ = geod.geometry_area_perimeter(gdf_tiles.iloc[0].geometry)
# area_geod = abs(area)
# #print(area/1e6)

In [33]:
gdf_tiles.head()

,name,id,tx,ty,zoom,geometry,path,ref_region,area_perc_org,area_perc_red,nearest_station_id,nearest_station_longitude,nearest_station_latitude,nearest_station_distance,intertidal_coverage,intertidal_coverage_ed,processed,tile_area,intertidal_area_ed
0,z10_x561_y116,2195,561.0,116.0,10,"POLYGON ((1662126.937 7226866.115, 1696047.895...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,2.32,1.18,station 07499,17.33421,79.92993,3.062257,8.286716,1.182026,True,47.352511,0.559719
1,z10_x667_y108,13635,667.0,108.0,10,"POLYGON ((5257748.475 7237575.520, 5291669.433...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,1.29,0.14,station 07502,54.76023,80.27526,12.472398,16.578475,0.135397,True,42.988448,0.058205
2,z10_x566_y141,2760,566.0,141.0,10,"POLYGON ((1831731.727 7185891.778, 1865652.685...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,0.71,0.07,station 07487,18.46672,77.95945,35.249050,2.918707,0.157026,True,63.989524,0.100480
3,z10_x545_y140,491,545.0,140.0,10,"POLYGON ((1119391.611 7187778.209, 1153312.569...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,22.14,21.76,station 07438,11.91355,78.40147,10.215705,22.184213,21.787025,True,63.225655,13.774989
7,z10_x550_y146,1037,550.0,146.0,10,"POLYGON ((1288996.400 7176112.427, 1322917.358...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,17.38,16.74,station 07491,13.86779,78.09662,25.418648,17.374234,16.777855,True,67.946233,11.399921


### 6. Calculate area based on sdb results (after post-processing, i.e. applying masks)

In [24]:
# # Reproject tiles back to WGS84
# gdf_tiles = gdf_tiles.to_crs('EPSG:4326')

# # Get area of the sdb files
# for i, row in tqdm(gdf_tiles.iterrows(), total=gdf_tiles.shape[0]):
#     # Read sdb file
#     ds = rxr.open_rasterio(row['file_path'])
    
#     # Clip to tile
#     ds = ds.rio.clip([row['geometry']])

#     # Get coverage of the tile
#     intertidal_coverage_sdb = ds.notnull().sum() / ds.size * 100
#     gdf_tiles.at[i, 'intertidal_coverage_sdb'] = intertidal_coverage_sdb
#     gdf_tiles.at[i, 'intertidal_area_sdb'] = gdf_tiles.at[i, 'tile_area'] * intertidal_coverage_sdb/100

### 7. Sum global coverage per region

In [28]:
# Group gdf by ref_region [OLD]
#gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed', 'intertidal_area_sdb']].groupby('ref_region').agg(
#    {'tile_area': 'sum', 'intertidal_area_ed': 'sum', 'tile_area': 'sum', 'intertidal_area_sdb': 'mean'}).reset_index()
gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed']].groupby('ref_region').agg(
    {'tile_area': 'sum', 'intertidal_area_ed': 'sum', 'tile_area': 'sum'}).reset_index()
gdf_tiles_grouped['tile_area'] = gdf_tiles_grouped['tile_area'].astype(int)
gdf_tiles_grouped['intertidal_area_ed'] = gdf_tiles_grouped['intertidal_area_ed'].astype(int)
#gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].astype(int)
#gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].round(2)

gdf_tiles_grouped

,ref_region,tile_area,intertidal_area_ed,intertidal_area_sdb
0,CAF,27568,900,7
1,CAU,41353,2129,9
2,EAO,7658,231,13
3,EAU,64327,2448,8
4,EIO,35226,829,1
5,ESAF,65859,2995,16
6,MDG,117933,5300,13
7,MED,188387,9215,20
8,NAU,168476,7775,10
9,NEAF,15316,413,16


In [25]:
# Group gdf by ref_region (FOR FILTERED TILES; 18797 along global coastline)
# TODO: ADD DATASET VALUES SDB LATER

#gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed', 'intertidal_area_sdb']].groupby('ref_region').agg(
#    {'tile_area': 'sum', 'intertidal_area_ed': 'sum', 'intertidal_area_sdb': 'mean'}).reset_index()
gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed']].groupby('ref_region').agg(
    {'tile_area': 'sum', 'intertidal_area_ed': 'sum'}).reset_index()
gdf_tiles_grouped['tile_area'] = gdf_tiles_grouped['tile_area'].astype(int)
gdf_tiles_grouped['intertidal_area_ed'] = gdf_tiles_grouped['intertidal_area_ed'].astype(int)
#gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].astype(int)
#gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].round(2)

# Add total row
total = pd.DataFrame([{
    "ref_region": "TOTAL",
    "tile_area": gdf_tiles_grouped["tile_area"].sum(),
    "intertidal_area_ed": gdf_tiles_grouped["intertidal_area_ed"].sum()
}])

# Append the total row
grouped_with_total = pd.concat([gdf_tiles_grouped, total], ignore_index=True)

# Add percentage column
grouped_with_total["intertidal_area_ed_perc"] = (grouped_with_total["intertidal_area_ed"] / grouped_with_total["tile_area"]) * 100
grouped_with_total["intertidal_area_ed_perc"] = grouped_with_total["intertidal_area_ed_perc"].round(2)

grouped_with_total

# Do sorting on different columns to explain differences in the table (analyse!)

,ref_region,tile_area,intertidal_area_ed,intertidal_area_ed_perc
0,ARO,37382,1282,3.43
1,ARP,275040,11669,4.24
2,ARS,5936,61,1.03
3,BOB,44176,677,1.53
4,CAF,68077,1074,1.58
5,CAR,330841,17702,5.35
6,CAU,61960,2279,3.68
7,CNA,46250,5643,12.20
8,EAN,53130,3445,6.48
9,EAO,10646,239,2.24


In [26]:
# for all tiles

# Reproject tiles to metres
gdf_tiles_org = gdf_tiles_org.to_crs('EPSG:6933') # Project to equal-area for grid-based metrics, EPSG 3857 is not to be used for area or dist (only mapping)

# Calculate area of the tiles
gdf_tiles_org['tile_area'] = gdf_tiles_org['geometry'].area / 1e6  # Convert to km2

# Calculate coverage
gdf_tiles_org['intertidal_area_ed'] = gdf_tiles_org['tile_area'] * gdf_tiles_org['intertidal_coverage_ed']/100

In [27]:
gdf_tiles_org.head()

,name,id,tx,ty,zoom,geometry,path,ref_region,area_perc_org,area_perc_red,nearest_station_id,nearest_station_longitude,nearest_station_latitude,nearest_station_distance,intertidal_coverage,intertidal_coverage_ed,tile_area,intertidal_area_ed
0,z10_x561_y116,2195,561.0,116.0,10,"POLYGON ((1662126.937 7226866.115, 1696047.895...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,2.32,1.18,station 07499,17.33421,79.92993,3.062257,8.286716,1.182026,47.352511,0.559719
1,z10_x667_y108,13635,667.0,108.0,10,"POLYGON ((5257748.475 7237575.520, 5291669.433...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,1.29,0.14,station 07502,54.76023,80.27526,12.472398,16.578475,0.135397,42.988448,0.058205
2,z10_x566_y141,2760,566.0,141.0,10,"POLYGON ((1831731.727 7185891.778, 1865652.685...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,0.71,0.07,station 07487,18.46672,77.95945,35.249050,2.918707,0.157026,63.989524,0.100480
3,z10_x545_y140,491,545.0,140.0,10,"POLYGON ((1119391.611 7187778.209, 1153312.569...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,22.14,21.76,station 07438,11.91355,78.40147,10.215705,22.184213,21.787025,63.225655,13.774989
4,z10_x644_y109,11152,644.0,109.0,10,"POLYGON ((4477566.443 7236292.788, 4511487.401...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,ARO,0.42,0.00,station 07509,46.96189,80.35473,7.708247,1.606905,0.000000,43.511501,0.000000


In [28]:
# Group gdf by ref_region (FOR ALL TILES; 38639 along global coastline)

#gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed', 'intertidal_area_sdb']].groupby('ref_region').agg(
#    {'tile_area': 'sum', 'intertidal_area_ed': 'sum', 'intertidal_area_sdb': 'mean'}).reset_index()
gdf_tiles_grouped_org = gdf_tiles_org[['ref_region', 'tile_area', 'intertidal_area_ed']].groupby('ref_region').agg(
    {'tile_area': 'sum', 'intertidal_area_ed': 'sum'}).reset_index()
gdf_tiles_grouped_org['tile_area'] = gdf_tiles_grouped_org['tile_area'].astype(int)
gdf_tiles_grouped_org['intertidal_area_ed'] = gdf_tiles_grouped_org['intertidal_area_ed'].astype(int)
#gdf_tiles_grouped_org['intertidal_area_sdb'] = gdf_tiles_grouped_org['intertidal_area_sdb'].astype(int)
#gdf_tiles_grouped_org['intertidal_area_sdb'] = gdf_tiles_grouped_org['intertidal_area_sdb'].round(2)

# Add total row
total_org = pd.DataFrame([{
    "ref_region": "TOTAL",
    "tile_area": gdf_tiles_grouped_org["tile_area"].sum(),
    "intertidal_area_ed": gdf_tiles_grouped_org["intertidal_area_ed"].sum()
}])

# Append the total row
grouped_with_total_org = pd.concat([gdf_tiles_grouped_org, total_org], ignore_index=True)

# Add percentage column
grouped_with_total_org["intertidal_area_ed_perc"] = (grouped_with_total_org["intertidal_area_ed"] / grouped_with_total_org["tile_area"]) * 100
grouped_with_total_org["intertidal_area_ed_perc"] = grouped_with_total_org["intertidal_area_ed_perc"].round(2)

grouped_with_total_org

# Do sorting on different columns to explain differences in the table (analyse!)

,ref_region,tile_area,intertidal_area_ed,intertidal_area_ed_perc
0,ARO,75856,1533,2.02
1,ARP,474295,14459,3.05
2,ARS,68251,602,0.88
3,BOB,66329,796,1.20
4,CAF,107177,1183,1.10
5,CAR,620288,20557,3.31
6,CAU,121182,3910,3.23
7,CNA,54382,5654,10.40
8,EAN,252529,5267,2.09
9,EAO,31898,252,0.79
